# Lab 23: Spectral Graph Theory — Independent Study Notebook

This notebook accompanies Chapter 23. It includes explanations, Python computations, practice questions, and worked answers.

Topics:

1. adjacency, degree, and Laplacian matrices;
2. powers of adjacency matrices and walk counting;
3. Laplacian energy;
4. incidence matrices;
5. connected components and zero eigenvalues;
6. Fiedler vectors and spectral clustering.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
np.set_printoptions(precision=4, suppress=True)

## 1. Build a graph and compute its matrices

We begin with two triangles connected by one bridge edge.

In [ ]:
G = nx.Graph()
G.add_edges_from([(0,1),(1,2),(2,0),
                  (3,4),(4,5),(5,3),
                  (2,3)])

pos = nx.spring_layout(G, seed=2)
nx.draw(G, pos, with_labels=True, node_color="lightblue", node_size=700, edge_color="gray")
plt.title("Two triangles connected by a bridge")
plt.show()

In [ ]:
A = nx.to_numpy_array(G, dtype=int)
D = np.diag(A.sum(axis=1))
L = D - A
print("Adjacency matrix A:
", A)
print("Degree matrix D:
", D)
print("Laplacian matrix L = D - A:
", L)

### Practice 1

Explain why $A$ is symmetric and why every row of $L$ sums to zero.

<details>
<summary><strong>Answer</strong></summary>

The graph is undirected, so if vertex $i$ is adjacent to vertex $j$, then vertex $j$ is adjacent to vertex $i$. Hence $A_{ij}=A_{ji}$.

For $L=D-A$, the diagonal entry $D_{ii}$ is the degree of vertex $i$, while the row sum of $A$ is also the degree of vertex $i$. Therefore the row sum of $L$ is $d_i-d_i=0$.

</details>


## 2. Powers of the adjacency matrix count walks

In [ ]:
A2 = A @ A
A3 = A2 @ A
print("A^2:
", A2)
print("A^3:
", A3)

The entry $(A^k)_{ij}$ counts walks of length $k$ from vertex $i$ to vertex $j$.

### Practice 2

In the matrix $A^2$ above, interpret the entry $(A^2)_{0,3}$ using graph language.

<details>
<summary><strong>Answer</strong></summary>

It counts the number of walks of length 2 from vertex 0 to vertex 3. In this graph there is one such walk: $0\to 2\to 3$.

</details>


## 3. Laplacian energy

In [ ]:
x = np.array([1, 1, 1, -1, -1, -1], dtype=float)
energy_matrix = x @ L @ x
energy_edges = sum((x[i]-x[j])**2 for i,j in G.edges())
print("x^T L x =", energy_matrix)
print("sum over edges (x_i-x_j)^2 =", energy_edges)

The identity is
$$
x^TLx=\sum_{\{i,j\}\in E}(x_i-x_j)^2.
$$

The chosen signal is constant on each triangle and changes only across the bridge. Thus most edges contribute zero energy.


## 4. Incidence matrix and the identity L = B B^T

In [ ]:
# Choose an arbitrary orientation for each edge.
edges = list(G.edges())
B = np.zeros((G.number_of_nodes(), len(edges)))
for k, (i, j) in enumerate(edges):
    B[i, k] = -1
    B[j, k] = 1

print("Edges:", edges)
print("Incidence matrix B:
", B)
print("B B^T:
", B @ B.T)
print("L:
", L)
print("Check:", np.allclose(B @ B.T, L))

### Practice 3

Why does changing the orientation of one edge not change $BB^T$?

<details>
<summary><strong>Answer</strong></summary>

Changing the orientation of one edge multiplies the corresponding column $b_e$ of $B$ by $-1$. But
$$
(-b_e)(-b_e)^T=b_e b_e^T.
$$
Therefore the sum $BB^T=\sum_e b_e b_e^T$ does not change.

</details>


## 5. Laplacian eigenvalues and connected components

In [ ]:
evals, evecs = np.linalg.eigh(L)
print("Laplacian eigenvalues:", evals)
print("Number of near-zero eigenvalues:", np.sum(np.isclose(evals, 0)))
print("NetworkX connected components:", list(nx.connected_components(G)))

The multiplicity of the zero eigenvalue equals the number of connected components.

Now create a disconnected graph.

In [ ]:
H = nx.Graph()
H.add_edges_from([(0,1),(1,2),(3,4),(4,5)])
LH = nx.laplacian_matrix(H).toarray()
evals_H = np.linalg.eigvalsh(LH)
print("Eigenvalues of disconnected graph:", evals_H)
print("Number of zero eigenvalues:", np.sum(np.isclose(evals_H, 0)))
print("Connected components:", list(nx.connected_components(H)))

## 6. Fiedler vector and spectral clustering

In [ ]:
evals, evecs = np.linalg.eigh(L)
fiedler = evecs[:, 1]
print("Eigenvalues:", evals)
print("Fiedler vector:", fiedler)

clusters = {"negative": np.where(fiedler < 0)[0].tolist(),
            "nonnegative": np.where(fiedler >= 0)[0].tolist()}
clusters

In [ ]:
plt.figure(figsize=(5,4))
nx.draw(G, pos, with_labels=True, node_color=fiedler, cmap="coolwarm", node_size=800, edge_color="gray")
plt.title("Fiedler vector coloring")
plt.show()

### Practice 4

What clustering is suggested by the Fiedler vector?

<details>
<summary><strong>Answer</strong></summary>

The sign pattern separates the two triangles. One cluster is approximately $\{0,1,2\}$ and the other is $\{3,4,5\}$.

</details>


## 7. Spectral embedding

Use the second and third Laplacian eigenvectors as coordinates.

In [ ]:
coords = evecs[:, 1:3]
plt.figure(figsize=(5,5))
for i, j in G.edges():
    plt.plot([coords[i,0], coords[j,0]], [coords[i,1], coords[j,1]], color="gray")
plt.scatter(coords[:,0], coords[:,1], s=120)
for i in range(len(coords)):
    plt.text(coords[i,0], coords[i,1], str(i), fontsize=12)
plt.axis("equal")
plt.title("Spectral embedding")
plt.show()

## 8. Similar practice problem

Create a path graph with 8 vertices. Compute its Laplacian eigenvalues and draw its spectral embedding.

In [ ]:
P = nx.path_graph(8)
LP = nx.laplacian_matrix(P).toarray()
valsP, vecsP = np.linalg.eigh(LP)
print("Path graph Laplacian eigenvalues:", valsP)

coordsP = vecsP[:, 1:3]
plt.figure(figsize=(5,5))
for i, j in P.edges():
    plt.plot([coordsP[i,0], coordsP[j,0]], [coordsP[i,1], coordsP[j,1]], color="gray")
plt.scatter(coordsP[:,0], coordsP[:,1], s=100)
for i in range(len(coordsP)):
    plt.text(coordsP[i,0], coordsP[i,1], str(i), fontsize=12)
plt.axis("equal")
plt.title("Spectral embedding of path graph P8")
plt.show()

<details>
<summary><strong>Reflection answer</strong></summary>

The second eigenvector changes smoothly from one end of the path to the other. This gives a natural one-dimensional ordering of the vertices along the path.

</details>


## 9. Final checklist

You should now be able to:

- build $A$, $D$, and $L$ from a graph;
- use $A^k$ to count walks;
- verify $x^TLx=\sum_{\{i,j\}}(x_i-x_j)^2$;
- use zero eigenvalues of $L$ to count connected components;
- use the Fiedler vector for a two-way spectral partition.
